In [2]:
%%bash
docker exec -i postgis-llm-eval-db psql -U postgres -d gis -c "\
    SELECT 'canada_census_divisions' AS table_name, \
        COUNT(*) AS row_count FROM public.canada_census_divisions \
        UNION ALL SELECT 'canada_airports', COUNT(*) FROM public.canada_airports \
        UNION ALL SELECT 'canada_roads', COUNT(*) FROM public.canada_roads \
        UNION ALL SELECT 'canada_religions_census_data', COUNT(*) FROM public.canada_religions_census_data;"

          table_name          | row_count 
------------------------------+-----------
 canada_census_divisions      |       293
 canada_airports              |        93
 canada_roads                 |   2242117
 canada_religions_census_data |       293
(4 rows)



In [5]:
%%bash
docker exec -i postgis-llm-eval-db psql -U postgres -d gis -c "\
    SELECT f_table_name, \
        f_geometry_column, \
        srid, \
        type \
    FROM public.geometry_columns \
    WHERE f_table_schema = 'public' \
    ORDER BY f_table_name;"

      f_table_name       | f_geometry_column | srid |      type       
-------------------------+-------------------+------+-----------------
 canada_airports         | geom              | 4269 | POINT
 canada_census_divisions | geom              | 3347 | MULTIPOLYGON
 canada_roads            | geom              | 3347 | MULTILINESTRING
(3 rows)



In [8]:
try:
    import psycopg
    connect = psycopg.connect
except ImportError:
    import psycopg2
    connect = psycopg2.connect

with connect(
    host="localhost",
    port=5433,
    dbname="gis",
    user="postgres",
    password="postgres",
) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT Count(*) FROM canada_roads;")
        road_count = cur.fetchone()[0]

print(f"canada_roads row count: {road_count}")

canada_roads row count: 2242117


In [7]:
!pip install psycopg2

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for psycopg2: filename=psycopg2-2.9.11-cp312-cp312-linux_x86_64.whl size=549451 sha256=6a527d1423910319c516455bbddea6f5901b0aa74d444412e08401ec572583f0
  Stored in directory: /home/codespace/.cache/pip/wheels/da/54/60/22d6c77229eaf3816b2a8baf906a85c5a8458f913249527d84
Successfully built psycopg2

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [11]:
try:
    import psycopg
    connect = psycopg.connect
except ImportError:
    import psycopg2
    connect = psycopg2.connect

describe_sql = """
SELECT
    ordinal_position,
    column_name,
    data_type,
    udt_name
FROM information_schema.columns
WHERE table_schema = 'public'
  AND table_name = 'canada_roads'
ORDER BY ordinal_position;
"""

with connect(
    host="localhost",
    port=5433,
    dbname="gis",
    user="postgres",
    password="postgres",
) as conn:
    with conn.cursor() as cur:
        cur.execute(describe_sql)
        road_columns = cur.fetchall()

for ordinal_position, column_name, data_type, udt_name in road_columns:
    print(f"{ordinal_position:>2}. {column_name:<20} {data_type} ({udt_name})")

 1. gid                  integer (int4)
 2. objectid             double precision (float8)
 3. ngd_uid              character varying (varchar)
 4. name                 character varying (varchar)
 5. type                 character varying (varchar)
 6. dir                  character varying (varchar)
 7. afl_val              character varying (varchar)
 8. atl_val              character varying (varchar)
 9. afr_val              character varying (varchar)
10. atr_val              character varying (varchar)
11. csddguid_l           character varying (varchar)
12. csduid_l             character varying (varchar)
13. csdname_l            character varying (varchar)
14. csdtype_l            character varying (varchar)
15. csddguid_r           character varying (varchar)
16. csduid_r             character varying (varchar)
17. csdname_r            character varying (varchar)
18. csdtype_r            character varying (varchar)
19. prdguid_l            character varying (varchar)
20. prui

In [19]:
from pprint import pprint
import psycopg2
connect = psycopg2.connect

def sql_query(query):
    with connect(
        host="localhost",
        port=5433,
        dbname="gis",
        user="postgres",
        password="postgres",
    ) as conn:
        with conn.cursor() as cur:
            cur.execute(query)
            return cur.fetchall()

#### 8. Simple SQL Exercises

**How many records are in the canada_roads table?**

In [22]:
names_sql = """
SELECT Count(*)
FROM canada_roads;
"""
sql_query(names_sql)


[(2242117,)]

**How many roads in Canada start with 'B'?**

In [27]:
query = """
SELECT COUNT(*)
FROM canada_roads
WHERE name LIKE 'B%';
"""
sql_query(query)

[(107238,)]

**What is the total Christian population in the Canada religion dataset?**

In [57]:
query = """
SELECT SUM(christian_total) AS christian_population
FROM canada_religions_census_data;
"""
sql_query(query)

[(19373295.0,)]

**What is the Christian population in Ontario?**

In [58]:
query = """
SELECT SUM(r.christian_total) AS christian_population
FROM canada_religions_census_data AS r
JOIN canada_census_divisions AS c
    ON c.dguid = r.dguid
WHERE c.pruid = '35';
"""
sql_query(query)

[(7315790.0,)]

**How many census divisions are in each province?**

In [59]:
query = """
SELECT pruid, COUNT(*) AS census_division_count
FROM canada_census_divisions
GROUP BY pruid
ORDER BY pruid;
"""
sql_query(query)

[('10', 11),
 ('11', 3),
 ('12', 18),
 ('13', 15),
 ('24', 98),
 ('35', 49),
 ('46', 23),
 ('47', 18),
 ('48', 19),
 ('59', 29),
 ('60', 1),
 ('61', 6),
 ('62', 3)]

**For each province, what percentage of the reported religion totals is no religion?**

In [60]:
query = """
SELECT
    c.pruid,
    100.0 * SUM(COALESCE(r.no_religion_total, 0)) / NULLIF(
        SUM(
            COALESCE(r.buddhist_total, 0) +
            COALESCE(r.christian_total, 0) +
            COALESCE(r.hindu_total, 0) +
            COALESCE(r.jewish_total, 0) +
            COALESCE(r.muslim_total, 0) +
            COALESCE(r.no_religion_total, 0) +
            COALESCE(r.other_religions_total, 0) +
            COALESCE(r.sikh_total, 0) +
            COALESCE(r.indigenous_spirituality_total, 0)
        ),
        0
    ) AS no_religion_pct
FROM canada_census_divisions AS c
JOIN canada_religions_census_data AS r
    ON c.dguid = r.dguid
GROUP BY c.pruid
ORDER BY c.pruid;
"""
sql_query(query)

[('10', 16.004661261727854),
 ('11', 28.462254120148856),
 ('12', 37.60246920038712),
 ('13', 29.655535796614636),
 ('24', 27.295804603615657),
 ('35', 31.597615677688882),
 ('46', 36.74784059002196),
 ('47', 36.61761173329949),
 ('48', 40.1189893445772),
 ('59', 52.06107553479743),
 ('60', 59.719590754073515),
 ('61', 39.78694413477022),
 ('62', 24.917987971569165)]

#### 10. Geometry Exercises

**What is the area of the Yukon census division?**

In [61]:
query = """
SELECT ST_Area(geom)
FROM canada_census_divisions
WHERE cdname = 'Yukon';
"""
sql_query(query)

[(456664827522.445,)]

**What is the geometry type of the road named '103 Street'? What is its length?**

In [62]:
query = """
SELECT
    ST_GeometryType(geom),
    ST_Length(geom)
FROM canada_roads
WHERE name = '103 Street';
"""
sql_query(query)

[('ST_MultiLineString', 76.38228948457095)]

**What is the GeoJSON representation of the 'Toronto City Centre' airport?**

In [63]:
query = """
SELECT ST_AsGeoJSON(geom)
FROM canada_airports
WHERE airport = 'Toronto City Centre';
"""
sql_query(query)

[('{"type":"Point","crs":{"type":"name","properties":{"name":"EPSG:4269"}},"coordinates":[-79.396101,43.627888]}',)]

**What is the total length of roads in Canada, in kilometers?**

In [ ]:
query = """
SELECT SUM(ST_Length(geom)) / 1000 AS total_km
FROM canada_roads;
"""
sql_query(query)

**What is the area of Ontario, in acres, based on its census divisions?**

In [ ]:
query = """
SELECT SUM(ST_Area(geom)) / 4046.8564224 AS acres
FROM canada_census_divisions
WHERE pruid = '35';
"""
sql_query(query)

**What is the most westerly airport in the dataset?**

In [ ]:
query = """
SELECT airport, ST_X(geom) AS longitude
FROM canada_airports
ORDER BY ST_X(geom)
LIMIT 1;
"""
sql_query(query)

**How long is the road named '103 Street'?**

In [28]:
query = """
SELECT ST_Length(geom)
FROM canada_roads
WHERE name = '103 Street';
"""
sql_query(query)

[(76.38228948457095,)]

**What is the length of roads in Canada, summarized by type?**

In [ ]:
query = """
SELECT type, SUM(ST_Length(geom)) AS length
FROM canada_roads
GROUP BY type
ORDER BY length DESC;
"""
sql_query(query)

#### 12. Spatial Relationships Exercises

**What is the geometry value for the road named '103 Street'?**

In [ ]:
query = """
SELECT ST_AsText(geom)
FROM canada_roads
WHERE name = '103 Street';
"""
sql_query(query)

**What census division is the road named '103 Street' in?**

In [ ]:
query = """
SELECT DISTINCT c.cdname, c.pruid
FROM canada_census_divisions AS c
JOIN canada_roads AS r
    ON ST_Intersects(c.geom, r.geom)
WHERE r.name = '103 Street';
"""
sql_query(query)

**What roads lie within 25 meters of the road named '103 Street'?**

In [39]:
query = """
SELECT DISTINCT r2.name
FROM canada_roads AS r1
JOIN canada_roads AS r2
    ON r1.gid <> r2.gid
   AND ST_DWithin(r1.geom, r2.geom, 25)
WHERE r1.name = 'Zenith'
  AND r2.name IS NOT NULL
ORDER BY r2.name;
"""
sql_query(query)

[('Aphelion',),
 ('Barons',),
 ('Birchmount',),
 ('Broad',),
 ('Cottonwood',),
 ('Cranbrook',),
 ('Dillingwood',),
 ('Larmere',),
 ('Lyrid',),
 ('Marsh',),
 ('Marta',),
 ('North Bonnington',),
 ('North Woodrow',),
 ('Orchid',),
 ('Patriot',),
 ('Railway',),
 ('Rippleton',),
 ('Riverbend',),
 ('Saskatchewan',),
 ('Silvio',),
 ('Zenith',)]

**Approximately how many people reporting Christianity live within 5 kilometers of the road named '103 Street'?**

In [40]:
query = """
SELECT SUM(d.christian_total)
FROM canada_census_divisions AS c
JOIN canada_religions_census_data AS d
    ON c.dguid = d.dguid
WHERE ST_DWithin(
    c.geom,
    (
        SELECT geom
        FROM canada_roads
        WHERE name = '103 Street'
        LIMIT 1
    ),
    5000
);
"""
sql_query(query)

[(57180.0,)]

#### 14. Spatial Joins Exercises

**What airport is in the Toronto census division? What type is it?**

In [41]:
query = """
SELECT a.airport, a.type
FROM canada_airports AS a
JOIN canada_census_divisions AS c
    ON ST_Contains(c.geom, ST_Transform(a.geom, 3347))
WHERE c.cdname = 'Toronto';
"""
sql_query(query)

[('Toronto City Centre', 'Control Tower')]

**What census divisions are served by Control Tower airports?**

In [42]:
query = """
SELECT DISTINCT c.cdname, c.pruid
FROM canada_airports AS a
JOIN canada_census_divisions AS c
    ON ST_Contains(c.geom, ST_Transform(a.geom, 3347))
WHERE a.type LIKE '%Control Tower%'
ORDER BY c.cdname;
"""
sql_query(query)

[('Algoma', '35'),
 ('Capital', '59'),
 ('Central Okanagan', '59'),
 ('Division No.  1', '10'),
 ('Division No. 11', '47'),
 ('Division No. 11', '48'),
 ('Division No. 13', '46'),
 ('Division No. 14', '46'),
 ('Division No. 16', '48'),
 ('Division No.  6', '10'),
 ('Division No.  6', '47'),
 ('Division No.  6', '48'),
 ('Durham', '35'),
 ('Essex', '35'),
 ('Fraser-Fort George', '59'),
 ('Fraser Valley', '59'),
 ('Greater Vancouver', '59'),
 ('Halifax', '12'),
 ('Hamilton', '35'),
 ('Le Haut-Richelieu', '24'),
 ('Le Saguenay-et-son-Fjord', '24'),
 ('Longueuil', '24'),
 ('Middlesex', '35'),
 ('Mirabel', '24'),
 ('Montréal', '24'),
 ('Ottawa', '35'),
 ('Peel', '35'),
 ('Québec', '24'),
 ('Region 6', '61'),
 ('Sunbury', '13'),
 ('Thunder Bay', '35'),
 ('Toronto', '35'),
 ('Waterloo', '35'),
 ('Westmorland', '13'),
 ('York', '35'),
 ('Yukon', '60')]

**How many people reporting no religion live in census divisions that contain at least one airport?**

In [43]:
query = """
SELECT SUM(d.no_religion_total)
FROM canada_religions_census_data AS d
JOIN (
    SELECT DISTINCT c.dguid
    FROM canada_census_divisions AS c
    JOIN canada_airports AS a
        ON ST_Contains(c.geom, ST_Transform(a.geom, 3347))
) AS airport_divisions
    ON airport_divisions.dguid = d.dguid;
"""
sql_query(query)

[(8715325.0,)]

**What census division has the highest Christian density, in persons per square kilometer?**

In [49]:
query = """
SELECT
    c.cdname,
    SUM(d.christian_total) / (ST_Area(c.geom) / 1000000.0) AS christian_per_sqkm
FROM canada_census_divisions AS c
JOIN canada_religions_census_data AS d
    ON c.dguid = d.dguid
GROUP BY c.cdname, c.geom
ORDER BY christian_per_sqkm DESC
LIMIT 1;
"""
sql_query(query)

[('Toronto', 1825.8130151779428)]

#### 17. Projection Exercises

**What is the length of all roads in Canada, as measured in SRID 3347?**

In [50]:
query = """
SELECT SUM(ST_Length(geom))
FROM canada_roads;
"""
sql_query(query)

[(1170167115.5945256,)]

**What is the WKT definition of SRID 3347?**

In [ ]:
query = """
SELECT srtext
FROM spatial_ref_sys
WHERE srid = 3347;
"""
sql_query(query)

**What is the length of all roads in Canada, as measured in SRID 3978?**

In [ ]:
query = """
SELECT SUM(ST_Length(ST_Transform(geom, 3978)))
FROM canada_roads;
"""
sql_query(query)

**How many roads cross the 123rd meridian?**

In [51]:
query = """
SELECT COUNT(*)
FROM canada_roads
WHERE ST_Intersects(
    ST_Transform(geom, 4326),
    'SRID=4326;LINESTRING(-123 40, -123 80)'::geometry
);
"""
sql_query(query)

[(170,)]

#### 19. Geography Exercises

**How far is Toronto City Centre from Vancouver International? What are the units of the answer?**

In [52]:
query = """
SELECT ST_Distance(a1.geom::geography, a2.geom::geography) AS meters
FROM canada_airports AS a1
CROSS JOIN canada_airports AS a2
WHERE a1.airport = 'Toronto City Centre'
  AND a2.airport = 'Vancouver International';
"""
sql_query(query)

[(3374064.94972115,)]

**What is the total length of all roads in Canada, calculated on the spheroid?**

In [53]:
query = """
SELECT SUM(ST_Length(Geography(ST_Transform(geom, 4326))))
FROM canada_roads;
"""
sql_query(query)

[(1170692451.042365,)]

**Does a point 40 meters north of Toronto City Centre intersect a 50 meter buffer around the airport in geography? In projected geometry?**

In [ ]:
query = """
WITH airport AS (
    SELECT geom, ST_Transform(geom, 3347) AS geom_3347
    FROM canada_airports
    WHERE airport = 'Toronto City Centre'
    LIMIT 1
)
SELECT
    ST_Intersects(
        ST_Project(geom::geography, 40, 0),
        ST_Buffer(geom::geography, 50)
    ) AS geography_intersects,
    ST_Intersects(
        ST_Translate(geom_3347, 0, 40),
        ST_Buffer(geom_3347, 50)
    ) AS geometry_intersects
FROM airport;
"""
sql_query(query)

#### 21. Geometry Constructing Exercises

**How many census divisions do not contain their own centroid?**

In [54]:
query = """
SELECT COUNT(*)
FROM canada_census_divisions
WHERE NOT ST_Contains(geom, ST_Centroid(geom));
"""
sql_query(query)

[(2,)]

**Union all census divisions into a single output. What kind of geometry is it? How many parts does it have?**

In [55]:
query = """
SELECT
    ST_GeometryType(geom) AS geometry_type,
    ST_NumGeometries(geom) AS num_parts
FROM (
    SELECT ST_Union(geom) AS geom
    FROM canada_census_divisions
) AS merged;
"""
sql_query(query)

[('ST_MultiPolygon', 3)]

**What is the area of a one-unit buffer around the origin?**

In [ ]:
query = """
SELECT ST_Area(ST_Buffer('POINT(0 0)'::geometry, 1));
"""
sql_query(query)

**Construct a 10 kilometer wide DMZ on the border between Toronto and Peel. What is the area of the DMZ?**

In [56]:
query = """
WITH dmz AS (
    SELECT ST_Intersection(
        ST_Buffer(a.geom, 5000),
        ST_Buffer(b.geom, 5000)
    ) AS geom
    FROM canada_census_divisions AS a, canada_census_divisions AS b
    WHERE a.cdname = 'Toronto'
      AND b.cdname = 'Peel'
 )
SELECT ST_Area(geom)
FROM dmz;
"""
sql_query(query)

[(336316736.1502637,)]